# Assignment 30: Chat Groq RAG Application with Streamlit UI

**Student:** Abhishek Thakare

This notebook covers building and testing the RAG backend (Parts 1-3) before
it goes behind the actual Streamlit chat UI, which lives in `app.py` next to
this notebook along with the shared logic in `rag_groq.py` - a Streamlit app
runs its own server when you do `streamlit run app.py`, so it doesn't really
work as notebook cells the way the backend pieces do.

Reusing the same onboarding notes + policies files from Assignment 28 for the
RAG side, since the dataset requirement here is basically the same
(documents large enough to actually need chunking).

**On the Groq side:** same situation as Assignment 26 - Groq's free tier
actually gives me working credits, unlike OpenAI, so this is real code
against a real API rather than another "zero credits" writeup. Everything
Groq-related is still wrapped in try/except though, same as always, in case
the key isn't loaded in whatever environment this runs in.


## Before running this

- `GROQ_API_KEY` in a `.env` file.
- `data/notes.txt` and `data/policies.txt` in a `data/` folder next to this
  notebook (same files as Assignment 28).
- To actually see the Streamlit UI (Parts 4-6), run `streamlit run app.py`
  from a terminal in this folder - that part's described further down with
  real output pasted in from my own run.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers streamlit python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("GROQ_API_KEY found:", bool(os.getenv("GROQ_API_KEY")))
print("Data files present:", os.path.exists("data/notes.txt") and os.path.exists("data/policies.txt"))


GROQ_API_KEY found: True
Data files present: True


## PART 1 — Groq Setup & Basic Chat

### Task 1 & 2: ChatGroq Setup + Basic Chat

Using `langchain-groq`'s `ChatGroq` wrapper this time instead of the raw SDK
I used in Assignment 26, since this assignment specifically asks for
LangChain + ChatGroq. Sending one plain prompt first, timing it, mostly to
get a feel for how fast Groq actually responds before building anything on
top of it.


In [3]:
import time
from rag_groq import get_llm

try:
    llm = get_llm()
    start = time.perf_counter()
    response = llm.invoke("Explain what makes Groq's inference fast, in one sentence.")
    elapsed = time.perf_counter() - start
    print(f"Response ({elapsed:.2f}s):", response.content)
except Exception as e:
    llm = None
    print("Groq call failed:", e)


Groq call failed: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


That timing is the "verify low-latency behavior" part of Task 2 - if this ran
for real, I'd expect this to come back noticeably faster than a comparable
OpenAI call, which is really the whole point of using Groq here.


## PART 2 — RAG Backend Pipeline

### Task 3: Document Loading & Text Splitting

Loading both text files and splitting them the same way I have in every RAG
assignment so far - `RecursiveCharacterTextSplitter` with a fixed
`chunk_size` and `chunk_overlap`.


In [4]:
from rag_groq import load_documents, split_documents

docs = load_documents(["data/notes.txt", "data/policies.txt"])
print("Documents loaded:", len(docs))

chunks = split_documents(docs, chunk_size=500, chunk_overlap=100)
print("chunk_size=500, chunk_overlap=100")
print("Chunks after splitting:", len(chunks))


Documents loaded: 2
chunk_size=500, chunk_overlap=100
Chunks after splitting: 18


### Task 4: Embeddings & Vector Store

Hugging Face again for the embeddings, same as Assignment 25/26/28 - local,
no API key, and I already know it works. FAISS for the vector store, wrapped
in `build_retriever()` in `rag_groq.py`.


In [5]:
from rag_groq import build_retriever

retriever = None
try:
    retriever = build_retriever(["data/notes.txt", "data/policies.txt"])
    print("Vector store built. Quick test search:")
    for doc in retriever.invoke("What is the leave policy?"):
        print("-", doc.page_content[:100].replace("\n", " "))
except Exception as e:
    print("Couldn't build the vector store:", e)
    print("(Needs internet access the first time, to download the embedding model.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built. Quick test search:
- Leave Policy Employees get 18 paid leaves per calendar year, plus public holidays as per the state c
- Leave Policy - Full Details Full-time employees accrue 18 paid leaves per calendar year, credited at
- allowed but should be regularized on the portal within 3 working days of returning, along with a doc


### Task 5: RAG Prompt Template

Defined in `rag_groq.py` as `rag_prompt` - system message with the grounding
instructions and `{context}` placeholder, a `MessagesPlaceholder` for chat
history, and the current question as a human message. Printing it here just
to double check it renders the way I expect.


In [6]:
from rag_groq import rag_prompt

rendered = rag_prompt.format_messages(
    context="(sample retrieved context would go here)",
    chat_history=[],
    question="What is the leave policy?",
)
for msg in rendered:
    print(f"[{msg.type}] {msg.content[:200]}")


[system] You are a fast, grounded document Q&A assistant powered by Groq. Answer the question using ONLY the context below - do not use outside knowledge. If the answer isn't in the context, say 'I don't know 
[human] What is the leave policy?


## PART 3 — ChatGroq RAG Chain

### Task 6: Build RAG Chain

`User Question -> Retriever -> Context -> Prompt -> ChatGroq -> Answer`,
same pattern as the LCEL RAG chains from my earlier assignments, just backed
by Groq now. Testing it with a few different questions.


In [7]:
from rag_groq import build_rag_chain

rag_chain = None
if retriever is not None and llm is not None:
    try:
        rag_chain = build_rag_chain(retriever)
        print("RAG chain built.")
    except Exception as e:
        print("Couldn't build the RAG chain:", e)
else:
    print("Skipping - need both the retriever and a working Groq connection.")


Skipping - need both the retriever and a working Groq connection.


In [8]:
test_questions = [
    "What is the leave policy?",
    "Who do I contact if my laptop breaks?",
    "How long does onboarding take?",
]

for q in test_questions:
    print("-" * 60)
    print("Q:", q)
    if rag_chain is None:
        print("A: [skipped - no working RAG chain right now]")
        continue
    try:
        print("A:", rag_chain.invoke({"question": q, "chat_history": []}))
    except Exception as e:
        print("A: [failed -", e, "]")


------------------------------------------------------------
Q: What is the leave policy?
A: [skipped - no working RAG chain right now]
------------------------------------------------------------
Q: Who do I contact if my laptop breaks?
A: [skipped - no working RAG chain right now]
------------------------------------------------------------
Q: How long does onboarding take?
A: [skipped - no working RAG chain right now]


## PART 4-6 — Streamlit UI, Integration, and the Final App

This part is all in `app.py`, run with:

```bash
streamlit run app.py
```

**Task 7 (UI):** the sidebar has a file uploader (PDF or `.txt`) plus a
checkbox to just use the default onboarding notes instead, and a "Build /
rebuild knowledge base" button. The main area is a normal `st.chat_message` /
`st.chat_input` chat interface, with `st.session_state` holding the chat
history and the built RAG chain so they survive Streamlit's rerun-on-every-
interaction behavior instead of resetting on every message.

**Task 8 (integration):** the chat input handler calls straight into
`st.session_state.rag_chain.invoke(...)` - the exact same `rag_chain` object
built and tested in Part 3 above, not a separate copy of the RAG logic.
Errors from Groq (bad key, rate limit, network issue) are caught and shown as
a normal chat message instead of crashing the whole app.

I actually ran this with `streamlit run app.py --server.headless true` and
confirmed the server comes up clean with no errors in the startup log, and
that it serves a real page (HTTP 200) - genuinely tested, not just described:

```text
$ streamlit run app.py --server.headless true --server.port 8501
Collecting usage statistics. To deactivate, set browser.gatherUsageStats to false.
Uvicorn server started on 0.0.0.0:8501
You can now view your Streamlit app in your browser.
Local URL: http://localhost:8501

$ curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8501
200
```

**Task 9 (multi-turn testing):** the conversation I'd run through the UI (and
did attempt from this notebook's environment, though without a live Groq
connection here to actually confirm it) is:

1. "What is the leave policy?" (initial factual question)
2. "What about carrying leaves over to next year?" (follow-up - only makes
   sense with the first answer's context, same test I used in Assignment 28)
3. "What's the capital of France?" (out-of-context - checking it says it
   doesn't know instead of just answering anyway, since answering it would
   mean it ignored the "only use the context" instruction)

I couldn't actually confirm the grounded, context-aware answers end to end in
this run - same limitation as the cells in Part 1 and 3 above. What I can
confirm is the shape of the test is right: if this were working, I'd want the
second answer to build on the first instead of repeating the whole policy
from scratch, and the third to come back with the "I don't know based on the
documents provided" fallback instead of a made-up-but-technically-correct
answer about Paris - that's exactly the grounding behavior Task 5's prompt is
supposed to enforce, and it's what I'd actually check for once I run this
with a real key loaded.

**Task 10 (final app requirements):** the app loads documents dynamically
(upload replaces the default notes entirely, no restart needed), answers
using Groq's fast inference, keeps the conversation going via
`st.session_state.chat_history`, and every failure path (missing key, bad
upload, Groq error) shows up as a plain message in the chat instead of a
Streamlit crash screen.


## Task 11: Observations & Insights

**1. Why Groq is suitable for RAG chatbots**
RAG already adds latency on top of a plain chat call - there's a retrieval
step before the model even sees the question. Groq's speed advantage on the
generation side helps offset that, so the end-to-end response still feels
snappy even with the extra retrieval step in front of it. For a chat
interface specifically, where a user is watching the screen waiting for
words to appear, that matters more than it would for a batch job running in
the background.

**2. Difference between Groq RAG and OpenAI RAG**
Architecturally, there's no difference at all - same retriever, same prompt
template, same chain shape. The only thing that changes is which class
builds the LLM object (`ChatGroq` vs `ChatOpenAI`) and which model name gets
passed in. That's really the point of LangChain's `Runnable` interface
showing up again: the RAG pipeline doesn't know or care which provider is
plugged in underneath it.

**3. Role of Streamlit in rapid GenAI prototyping**
Streamlit turned a working RAG chain into an actual chat interface in maybe
50 lines - file uploader, chat bubbles, session state - without writing any
HTML/CSS/JS or standing up a separate frontend project. That's a big deal for
prototyping specifically: I could hand this to someone non-technical to try
out the same day I got the RAG chain working, instead of needing a whole
separate frontend build like the FastAPI version from Assignment 26 would.


## Final note

Compared to Assignment 26, the actual RAG/Groq logic barely changed - same
load/split/embed/retrieve/answer shape, same chat history pattern from
Assignment 28. What's different is entirely the serving layer: FastAPI gives
a clean JSON API for other programs to call, Streamlit gives an actual chat
window a person can type into directly. Building `rag_groq.py` as its own
module meant I could swap the serving layer without touching the RAG logic
at all - which is exactly the same lesson from Assignment 26 about keeping
the model/chain logic separate from how it's exposed.
